# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Content Refresh Prioritization


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

My lane is **content refresh prioritization** for FlyRank. Editors can only refresh a handful of pages each week, but a client library holds thousands of published articles. I want to build a model that ranks pages by how much they need a refresh, so an editor's limited time goes to the pages where it matters most. I chose this lane because it uses the real FlyRank starter data, it maps cleanly onto a ranking/scoring task, and the "which page first?" decision has an obvious owner and a measurable payoff.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

My research question is: **which published pages should an editor refresh first?** The output is a ranked "refresh review" queue. The person who acts on it is a **content editor**, who works down the list and updates the top pages. A wrong call has two costs: a **false positive** wastes an editor's hours on a page that was fine, and a **false negative** lets a genuinely declining page keep losing traffic unreviewed. Because editor time is the scarce resource, I care most about the top of the list being trustworthy, so I'll measure the queue with **precision@K** (are the top-K flagged pages actually declining?).


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [7]:
import os, sys, subprocess
import pandas as pd

# --- Bootstrap: on Colab, clone the repo so the starter CSV is available ---
REPO_URL = "https://github.com/thany-8/content-refresh-prioritizer"
REPO_DIR = "content-refresh-prioritizer"
CSV = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(CSV):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to repo root when running locally

assert os.path.exists(CSV), "starter CSV not found — are you at the repo root?"
df = pd.read_csv(CSV)

# --- 3 real numbers that make the content-refresh lane worth 7 weeks ---
# 1. The size of the problem
n_pages = len(df)
n_clients = df["client_id"].nunique()

# 2. How common is decline? (trend_direction is the LABEL SOURCE — inspected here, never a feature)
declining = df["trend_direction"].eq("down")
decline_rate = declining.mean() * 100

# 3. Of declining pages, how many still earn real search traffic (worth an editor's time)?
worth_fixing = declining & (df["impressions_90d"] >= 100)
share_worth_fixing = worth_fixing.sum() / declining.sum() * 100

print(f"1. {n_pages:,} content items across {n_clients} clients — too many to review by hand.")
print(f"2. {decline_rate:.1f}% of pages are declining (trend_direction == 'down').")
print(f"3. {share_worth_fixing:.1f}% of declining pages still get >=100 impressions/90d — real traffic to save.")


1. 30,000 content items across 32 clients — too many to review by hand.
2. 54.2% of pages are declining (trend_direction == 'down').
3. 80.9% of declining pages still get >=100 impressions/90d — real traffic to save.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**I can say:** the model produces a **decision-support** ranking — a prioritized refresh queue measured by precision@K on **observed** decline in our own 90-day windows. Results are **directional**: pages the model ranks high tend to be ones that have been losing impressions.

**I can never say:** that the model **predicts what Google will do**, or that refreshing a page **causes** recovery. The label is a rule we compute from our own traffic (`trend_direction`), not a Google signal or a controlled experiment — so no causal proof, and no "predicting the algorithm".


In [9]:
# Why my claims must stay careful: the target is a RULE on our own data, not a Google signal.
# is_declining_label == (trend_direction == "down"), and trend_direction comes from trend_pct.
# So trend_direction and trend_pct can never be features (that would be leakage), and the model
# learns "which pages look like our declining pages", not "what Google will do next".
label_share = df["trend_direction"].eq("down").mean() * 100
print(f"Base rate of 'down' pages: {label_share:.1f}%  ->  a useful queue must beat this by ranking, not just guessing.")


Base rate of 'down' pages: 54.2%  ->  a useful queue must beat this by ranking, not just guessing.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.